In [2]:
from pathlib import Path

import pandas as pd


DATA_FILENAME = "customer_booking.csv"
DATA_FILE = Path.cwd() / "data" / DATA_FILENAME
if not DATA_FILE.exists():
    DATA_FILE = Path.cwd().parent / "data" / DATA_FILENAME


def load_customer_bookings() -> pd.DataFrame:
    """Load customer booking data from the data folder."""
    return pd.read_csv(DATA_FILE)

def load_customer_bookings() -> pd.DataFrame:
    """Load customer booking data from the data folder."""
    return pd.read_csv(DATA_FILE, encoding="latin-1")


df = load_customer_bookings()
df.head()

,num_passengers,sales_channel,trip_type,purchase_lead,length_of_stay,flight_hour,flight_day,route,booking_origin,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,booking_complete
0,2,Internet,RoundTrip,262,19,7,Sat,AKLDEL,New Zealand,1,0,0,5.52,0
1,1,Internet,RoundTrip,112,20,3,Sat,AKLDEL,New Zealand,0,0,0,5.52,0
2,2,Internet,RoundTrip,243,22,17,Wed,AKLDEL,India,1,1,0,5.52,0
3,1,Internet,RoundTrip,96,31,4,Sat,AKLDEL,New Zealand,0,0,1,5.52,0
4,2,Internet,RoundTrip,68,22,15,Wed,AKLDEL,India,1,0,1,5.52,0


In [4]:
df["booking_complete"].value_counts()

booking_complete
0    42522
1     7478
Name: count, dtype: int64

In [7]:
missing = df.isnull().sum()
missing

num_passengers           0
sales_channel            0
trip_type                0
purchase_lead            0
length_of_stay           0
flight_hour              0
flight_day               0
route                    0
booking_origin           0
wants_extra_baggage      0
wants_preferred_seat     0
wants_in_flight_meals    0
flight_duration          0
booking_complete         0
dtype: int64

In [8]:
df.dtypes

num_passengers             int64
sales_channel                str
trip_type                    str
purchase_lead              int64
length_of_stay             int64
flight_hour                int64
flight_day                   str
route                        str
booking_origin               str
wants_extra_baggage        int64
wants_preferred_seat       int64
wants_in_flight_meals      int64
flight_duration          float64
booking_complete           int64
dtype: object

In [9]:
df["route"].nunique()

799

In [12]:
df["route"].groupby(df["route"]).size().sort_values(ascending=False).head(10)

route
AKLKUL    2680
PENTPE     924
MELSGN     842
ICNSIN     801
DMKKIX     744
ICNSYD     695
DMKPER     679
DPSICN     666
DMKOOL     655
MELPEN     649
Name: route, dtype: int64

In [16]:
df["origin"] = df["route"].str[:3]
df["destination"] = df["route"].str[3:]

df[["route", "origin", "destination"]].head(5)

,route,origin,destination
0,AKLDEL,AKL,DEL
1,AKLDEL,AKL,DEL
2,AKLDEL,AKL,DEL
3,AKLDEL,AKL,DEL
4,AKLDEL,AKL,DEL


In [29]:
df["origin"].value_counts().head(15)


origin
DMK    3747
ICN    3732
MEL    3468
DPS    2902
AKL    2849
HKT    2387
CGK    2339
PEN    2111
HND    1774
COK    1673
PER    1453
KIX    1296
CMB    1289
BKI    1286
DEL    1205
Name: count, dtype: int64

In [28]:
df["destination"].value_counts().head(15)

destination
SYD    6616
PER    4992
MEL    4444
TPE    4044
OOL    2796
KUL    2688
PEN    2483
KIX    2240
ICN    2213
SIN    2157
SGN    1696
PVG    1575
PEK    1338
HND    1212
PUS    1038
Name: count, dtype: int64

In [31]:
df["booking_origin"].value_counts().head(15)

booking_origin
Australia        17872
Malaysia          7174
South Korea       4559
Japan             3885
China             3387
Indonesia         2369
Taiwan            2077
Thailand          2030
India             1270
New Zealand       1074
Singapore         1044
United States      456
Vietnam            388
Macau              304
Hong Kong          295
Name: count, dtype: int64

In [33]:
columns_to_encode = ["booking_origin", "destination", "origin"]

df_ohe = df.drop(columns=columns_to_encode).copy()

for col in columns_to_encode:
    top_values = df[col].value_counts().head(18).index
    grouped = df[col].where(df[col].isin(top_values), "Other")
    encoded = pd.get_dummies(grouped, prefix=col, dtype=int)
    df_ohe = pd.concat([df_ohe, encoded], axis=1)

df_ohe.head()

,num_passengers,sales_channel,trip_type,purchase_lead,length_of_stay,flight_hour,flight_day,route,wants_extra_baggage,wants_preferred_seat,...,origin_HND,origin_ICN,origin_JHB,origin_KCH,origin_KIX,origin_MEL,origin_OOL,origin_Other,origin_PEN,origin_PER
0,2,Internet,RoundTrip,262,19,7,Sat,AKLDEL,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1,Internet,RoundTrip,112,20,3,Sat,AKLDEL,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,Internet,RoundTrip,243,22,17,Wed,AKLDEL,1,1,...,0,0,0,0,0,0,0,0,0,0
3,1,Internet,RoundTrip,96,31,4,Sat,AKLDEL,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2,Internet,RoundTrip,68,22,15,Wed,AKLDEL,1,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_ohe = pd.get_dummies(
    df_ohe,
    columns=["sales_channel", "trip_type"],
    dtype=int
)

df_ohe.head()